# 1. Purpose and validation boundary

This Kaggle-first notebook consumes the canonical `ablation_summary.csv` produced by the repository aggregator. It does not run inference, retrieval evaluation, or live provider calls, and it does not infer a final pipeline from incomplete evidence. Copy this notebook to Kaggle, edit the next cell, and run from the top.

## 2. Editable Kaggle configuration

Replace the repository URL and branch. For existing artifacts, attach a Kaggle Dataset and optionally set the source paths. To aggregate attached run directories instead, set `RUN_AGGREGATOR=True` and `RUNS_SOURCE_DIR`; generated aggregates are written only beneath `/kaggle/working`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<OWNER>/<REPOSITORY>.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/TextMining")

FORCE_RECLONE = False
PULL_IF_EXISTS = True
INSTALL_DEPENDENCIES = True

RUNS_SOURCE_DIR = None
AGGREGATE_INPUT_DIR = None

RUN_AGGREGATOR = False

SUMMARY_CSV_SOURCE = None
REPORT_MD_SOURCE = None

OUTPUT_ROOT = Path("/kaggle/working/ablation_report_outputs")

PRIMARY_QUALITY_METRIC = "token_f1"
PRIMARY_LATENCY_METRIC = "average_total_latency_ms"

INCLUDE_INELIGIBLE_IN_FAMILY_TABLES = False
CREATE_EXPORT_ZIP = True

## 3. Kaggle filesystem detection

In [ ]:
import os
import shutil
import subprocess
import sys

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError("This notebook is designed for Kaggle. Create /kaggle/working or run it in a Kaggle session.")
print({"kaggle_input_available": KAGGLE_INPUT.is_dir(), "kaggle_working": str(KAGGLE_WORKING)})

## 4. Clone or update repository

In [ ]:
def run_checked(args, cwd=None):
    return subprocess.run(args, cwd=cwd, check=True, text=True, capture_output=True)

if "<OWNER>" in REPO_URL or "<REPOSITORY>" in REPO_URL:
    raise ValueError("Update REPO_URL in the editable configuration cell before running.")
if FORCE_RECLONE and REPO_DIR.exists():
    if REPO_DIR.resolve() in {Path("/").resolve(), KAGGLE_WORKING.resolve()}:
        raise RuntimeError("Refusing to remove a broad directory.")
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run_checked(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
elif (REPO_DIR / ".git").is_dir() and PULL_IF_EXISTS:
    run_checked(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_checked(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
required_checkout_paths = [REPO_DIR / "scripts/aggregate_ablation_results.py", REPO_DIR / "configs/ablation_configs.yaml", REPO_DIR / "src"]
missing_checkout_paths = [str(path) for path in required_checkout_paths if not path.exists()]
if missing_checkout_paths:
    raise RuntimeError(f"Repository checkout is invalid; missing: {missing_checkout_paths}")

## 5. Set repository root and imports

In [ ]:
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
REPOSITORY_BRANCH = run_checked(["git", "branch", "--show-current"], cwd=REPO_DIR).stdout.strip()
REPOSITORY_COMMIT = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
print({"repository_path": str(REPO_DIR), "branch": REPOSITORY_BRANCH, "commit": REPOSITORY_COMMIT})

## 6. Install or validate minimal dependencies

In [ ]:
import importlib.util

required_packages = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "yaml": "PyYAML"}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages and INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)
    raise RuntimeError("Dependencies were installed. Restart the Kaggle session and rerun from the first cell.")
if missing_packages:
    raise RuntimeError(f"Missing packages: {missing_packages}. Enable INSTALL_DEPENDENCIES or install them, then restart and rerun.")
try:
    import pandas as pd
except ImportError as exc:
    raise RuntimeError("pandas is unavailable. Install dependencies, restart the Kaggle session, and rerun from the first cell.") from exc
print({"pandas": pd.__version__, "gpu_required": False})

## 7. Locate aggregate inputs

In [ ]:
def explicit_path(value):
    return Path(value).expanduser() if value else None

def find_named_artifact(filename):
    roots = [explicit_path(AGGREGATE_INPUT_DIR), KAGGLE_INPUT, KAGGLE_WORKING, REPO_DIR]
    for root in [item for item in roots if item and item.exists()]:
        direct = root / filename
        if direct.is_file():
            return direct
        matches = sorted(root.rglob(filename))
        if matches:
            return matches[0]
    return None

summary_csv_path = explicit_path(SUMMARY_CSV_SOURCE) or find_named_artifact("ablation_summary.csv")
report_md_path = explicit_path(REPORT_MD_SOURCE) or find_named_artifact("ablation_report.md")
print({"summary_csv": str(summary_csv_path) if summary_csv_path else None, "supplementary_report": str(report_md_path) if report_md_path else None})

## 8. Optionally invoke canonical aggregator

In [ ]:
from scripts.aggregate_ablation_results import aggregate_ablation_results
from scripts.run_ablation_config import load_ablation_configs
from evaluation.ablation_analysis import (
    FAMILIES, add_baseline_deltas, build_family_table, classify_summary,
    compute_pareto_frontier, diagnostic_rows, expected_configs_by_family,
    export_analysis_outputs, generate_mechanical_observations,
    load_ablation_summary, select_comparable_rows, summarize_coverage,
)

if RUN_AGGREGATOR:
    if RUNS_SOURCE_DIR is None:
        raise ValueError("Set RUNS_SOURCE_DIR before enabling RUN_AGGREGATOR.")
    runs_source = Path(RUNS_SOURCE_DIR)
    if not runs_source.is_dir():
        raise FileNotFoundError(f"Run directory not found: {runs_source}")
    aggregate_output = KAGGLE_WORKING / "ablation_aggregate_inputs"
    if aggregate_output.exists():
        raise FileExistsError(f"Refusing to overwrite aggregate output: {aggregate_output}")
    aggregate_output.mkdir(parents=True)
    summary_csv_path = aggregate_output / "ablation_summary.csv"
    report_md_path = aggregate_output / "ablation_report.md"
    aggregate_ablation_results(runs_source, output_csv=summary_csv_path, output_report=report_md_path)
if summary_csv_path is None or not summary_csv_path.is_file():
    raise FileNotFoundError("No canonical ablation_summary.csv was found. Attach aggregate outputs as a Kaggle Dataset and set SUMMARY_CSV_SOURCE, or enable RUN_AGGREGATOR with RUNS_SOURCE_DIR.")

## 9. Load and validate ablation_summary.csv

In [ ]:
loaded = load_ablation_summary(summary_csv_path)
summary = loaded.frame
if loaded.validation.missing_required_columns:
    raise ValueError("Canonical aggregate schema is incomplete: " + "; ".join(loaded.validation.messages()))
config_registry = load_ablation_configs(REPO_DIR / "configs/ablation_configs.yaml")
config_metadata = {name: config.get("metadata", {}) for name, config in config_registry.items()}
expected_configs = expected_configs_by_family(config_registry)
summary = classify_summary(summary, config_metadata)
display(pd.DataFrame({"schema_valid": [loaded.validation.is_valid], "diagnostic": ["; ".join(loaded.validation.messages()) or "No schema diagnostics"]}))
display(pd.DataFrame({"column": summary.columns, "dtype": [str(dtype) for dtype in summary.dtypes]}))

## 10. Show dataset and run coverage

In [ ]:
coverage = summarize_coverage(summary, expected_configs)
display(coverage)
family_availability = coverage.loc[coverage["measure"].isin(["available_config_count", "eligible_completed_rows", "missing_expected_configs"])]
display(family_availability)

## 11. Show family-specific comparisons

In [ ]:
family_tables = {}
for family in FAMILIES[:-1]:
    table = build_family_table(summary, family, INCLUDE_INELIGIBLE_IN_FAMILY_TABLES)
    table = add_baseline_deltas(table, family)
    family_tables[family] = table
    print(f"{family.upper()} ablation table ({len(table)} row(s))")
    display(table.fillna("N/A"))

## 12. Show quality-latency analysis

In [ ]:
pareto_candidates = compute_pareto_frontier(summary, PRIMARY_QUALITY_METRIC, PRIMARY_LATENCY_METRIC)
print({"quality_metric": PRIMARY_QUALITY_METRIC, "latency_metric": PRIMARY_LATENCY_METRIC, "latency_unit": "ms"})
display(pareto_candidates.fillna("N/A"))
print("Plots are generated during export only when at least two eligible comparable rows contain both selected metrics.")

## 13. Show failed, excluded, and deferred runs

In [ ]:
excluded_runs, failed_deferred_runs = diagnostic_rows(summary)
print("Excluded, incompatible, or otherwise ineligible rows")
display(excluded_runs.fillna("N/A"))
print("Failed, skipped, deferred, partial, needs-rerun, and invalid rows")
display(failed_deferred_runs.fillna("N/A"))
missing_metrics = summary.loc[summary[[column for column in [PRIMARY_QUALITY_METRIC, PRIMARY_LATENCY_METRIC] if column in summary]].isna().any(axis=1)]
display(missing_metrics[[column for column in ["run_id", "config_name", "family", "status", PRIMARY_QUALITY_METRIC, PRIMARY_LATENCY_METRIC, "notes"] if column in missing_metrics]].fillna("N/A"))

## 14. Generate report-ready analysis

In [ ]:
mechanical_observations = generate_mechanical_observations(summary, PRIMARY_QUALITY_METRIC, PRIMARY_LATENCY_METRIC)
for observation in mechanical_observations:
    print(f"- {observation}")
if report_md_path and report_md_path.is_file():
    print("The canonical Markdown report is available as supplementary diagnostics; its values are not used to rebuild comparisons.")

## 15. Export tables, figures, and notes to /kaggle/working

In [ ]:
export_directory = export_analysis_outputs(
    summary, OUTPUT_ROOT, summary_csv_path, REPOSITORY_COMMIT,
    quality_metric=PRIMARY_QUALITY_METRIC, latency_metric=PRIMARY_LATENCY_METRIC,
    expected=expected_configs, create_zip=CREATE_EXPORT_ZIP,
)
print(f"Exported analysis artifacts to: {export_directory}")
for artifact in sorted(export_directory.iterdir()):
    print(f"- {artifact.name}")
if CREATE_EXPORT_ZIP:
    print(f"- {export_directory.with_suffix('.zip')}")